In [1]:
# import fastf1
import pandas as pd
import os
import sys
import importlib

sys.path.append(os.path.abspath(".."))

import src.fantasy
import src.models
import src.features

importlib.reload(src.fantasy)
importlib.reload(src.models)
importlib.reload(src.features)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from src.fantasy import build_fantasy_team, generate_explanations, evaluate_fantasy_picks
from src.data_loader import save_dataset, load_dataset
from src.features import build_features
from src.models import prepare_data, train_model, predict, build_fantasy_table, evaluate_model, tune_gainer_weights
# fastf1.Cache.enable_cache("../data/cache")

In [2]:
df_2023 = load_dataset("../data/processed/race_results_2023.csv")
df_2024 = load_dataset("../data/processed/race_results_2024.csv")

df = pd.concat([df_2023, df_2024], ignore_index=True)

In [3]:
df_2023 = build_features(df_2023)
df_2024 = build_features(df_2024)

In [4]:
X_train, y_train, X_test, y_test = prepare_data(df_2023, df_2024)

model = train_model(X_train, y_train)
predictions = predict(model,X_test)

fantasy_table = build_fantasy_table(df_2024, predictions)

In [5]:
team = build_fantasy_team(fantasy_table, "Canadian Grand Prix")
team[["Abbreviation", "GridPosition", "Predicted", "PositionChange", "FantasyValue", "PickCategory", "GainerScore"]]


,Abbreviation,GridPosition,Predicted,PositionChange,FantasyValue,PickCategory,GainerScore
179,VER,2.0,1.85,1.0,1.2,Safe,NaN
180,NOR,3.0,2.24,1.0,1.2,Safe,NaN
181,RUS,1.0,3.25,-2.0,-0.3,Safe,NaN
182,HAM,7.0,6.06,3.0,2.2,Value,-0.157333
184,ALO,6.0,6.61,0.0,0.4,Avoid,-0.899333
183,PIA,4.0,5.15,-1.0,0.2,Risk,-1.090000


In [6]:
team = build_fantasy_team(fantasy_table, "Qatar Grand Prix")
team = generate_explanations(team)
team[["Abbreviation", "PickCategory", "Explanation"]]

,Abbreviation,PickCategory,Explanation
459,VER,Safe,"Safe Pick: starts P2, predicted P1, consistent..."
460,LEC,Safe,"Safe Pick: starts P5, predicted P3, consistent..."
461,PIA,Safe,"Safe Pick: starts P4, predicted P3, consistent..."
464,SAI,Risk,"Risk Pick: starts p7, predicted P5, inconsiste..."
466,ZHO,Value,"Value Pick: starts P12, predicted P10, average..."
463,GAS,Value,"Value Pick: starts P11, predicted P10, average..."


In [7]:
# best_weights = tune_gainer_weights(fantasy_table)
# best_weights

In [8]:
v4_results = evaluate_fantasy_picks(fantasy_table)

v4_results

,RaceName,Total Picks,Top 5 Hits,Top 5 Hit Rate,Top 10 Hits,Top 10 Hit Rate,Avg Finish,Avg Position Gain,DNF Count
0,Miami Grand Prix,6,3,0.50,5,0.83,5.33,0.33,0
1,British Grand Prix,6,4,0.67,6,1.00,4.33,1.33,0
2,Belgian Grand Prix,6,3,0.50,5,0.83,5.83,1.00,0
3,Italian Grand Prix,6,3,0.50,6,1.00,5.17,0.83,0
4,Azerbaijan Grand Prix,6,3,0.50,5,0.83,5.33,0.83,0
5,Australian Grand Prix,4,3,0.75,4,1.00,3.50,1.25,0
6,Japanese Grand Prix,6,3,0.50,6,1.00,4.67,0.00,0
7,Chinese Grand Prix,6,3,0.50,5,0.83,6.17,-0.33,0
8,Canadian Grand Prix,6,5,0.83,6,1.00,3.50,0.33,0
9,Hungarian Grand Prix,6,4,0.67,6,1.00,5.17,-0.17,0


In [9]:
v3_results = evaluate_fantasy_picks(fantasy_table,
    w_avg_position_change=0.6,
    w_predicted=0.2,
    w_confidence=0.1,
    w_grid_gap=0.4
)

v3_results

,RaceName,Total Picks,Top 5 Hits,Top 5 Hit Rate,Top 10 Hits,Top 10 Hit Rate,Avg Finish,Avg Position Gain,DNF Count
0,Miami Grand Prix,6,3,0.50,5,0.83,5.33,0.33,0
1,British Grand Prix,6,4,0.67,6,1.00,4.33,1.33,0
2,Belgian Grand Prix,6,3,0.50,5,0.83,5.83,1.00,0
3,Italian Grand Prix,6,3,0.50,6,1.00,5.17,0.83,0
4,Azerbaijan Grand Prix,6,3,0.50,5,0.83,5.33,0.83,0
5,Australian Grand Prix,4,3,0.75,4,1.00,3.50,1.25,0
6,Japanese Grand Prix,6,3,0.50,6,1.00,4.67,0.00,0
7,Chinese Grand Prix,6,3,0.50,5,0.83,6.17,-0.33,0
8,Canadian Grand Prix,6,5,0.83,6,1.00,3.50,0.33,0
9,Hungarian Grand Prix,6,4,0.67,6,1.00,5.17,-0.17,0


In [10]:
team_v4 = build_fantasy_team(fantasy_table, "Qatar Grand Prix")
team_v4[["Abbreviation", "PickCategory", "GainerScore", "PositionChange"]]

,Abbreviation,PickCategory,GainerScore,PositionChange
459,VER,Safe,NaN,1.0
460,LEC,Safe,NaN,3.0
461,PIA,Safe,NaN,1.0
464,SAI,Risk,0.052000,1.0
466,ZHO,Value,-0.185333,4.0
463,GAS,Value,-0.234667,6.0


In [11]:
team_v3 = build_fantasy_team(fantasy_table, "Qatar Grand Prix",
    w_avg_position_change=0.6,
    w_predicted=0.2,
    w_confidence=0.1,
    w_grid_gap=0.4
)
team_v3[["Abbreviation", "PickCategory", "GainerScore", "PositionChange"]]

,Abbreviation,PickCategory,GainerScore,PositionChange
459,VER,Safe,NaN,1.0
460,LEC,Safe,NaN,3.0
461,PIA,Safe,NaN,1.0
463,GAS,Value,0.465,6.0
466,ZHO,Value,0.135,4.0
465,ALO,Risk,-2.261,1.0


In [12]:
v4_results = evaluate_fantasy_picks(fantasy_table)

v3_results = evaluate_fantasy_picks(fantasy_table,
    w_avg_position_change=0.6,
    w_predicted=0.2,
    w_confidence=0.1,
    w_grid_gap=0.4
)

In [14]:
print("V4 Results")
print(v4_results[["Top 5 Hit Rate", "Top 10 Hit Rate", "Avg Finish", "Avg Position Gain"]].mean().round(2))

print("V3 Results")
print(v3_results[["Top 5 Hit Rate", "Top 10 Hit Rate", "Avg Finish", "Avg Position Gain"]].mean().round(2))

V4 Results
Top 5 Hit Rate       0.61
Top 10 Hit Rate      0.95
Avg Finish           4.68
Avg Position Gain    0.54
dtype: float64
V3 Results
Top 5 Hit Rate       0.61
Top 10 Hit Rate      0.95
Avg Finish           4.71
Avg Position Gain    0.55
dtype: float64


In [16]:
save_dataset(v4_results, "../results/fantasy_v4_results.csv")